In [ ]:
#@title 1. 준비 : Google Drive 마운트, 코드 받기, 실행 환경 설치 {display-mode: "form"}
#@markdown 처음 한 번 1~2분 걸림. 같은 세션에서는 다시 실행해도 금방 끝남.
branch = 'turbo' #@param {type:"string"}

import os
from google.colab import drive

REPO = '/content/whisper-am'

drive.mount ('/content/drive')

if os.path.isdir (REPO) :
    !git -C {REPO} pull -q
else :
    !git clone -q -b {branch} https://github.com/bhnatt/whisper-am.git {REPO}

!bash {REPO}/colab/setup.sh


In [ ]:
#@title 2. 실행 : 오디오 → srt, txt, docx {display-mode: "form"}
#@markdown 아래 출력되는 터미널 커맨드를 Colab 터미널에 붙여넣어도 똑같이 동작함.
data_dir = '/content/drive/MyDrive/data' #@param {type:"string"}
out_dir = '' #@param {type:"string"}
#@markdown out_dir 비워두면 data_dir에 저장
model = 'guillaumekln/faster-whisper-large-v2' #@param ["guillaumekln/faster-whisper-large-v2", "Systran/faster-whisper-large-v3", "Systran/faster-whisper-medium.en", "tiny.en"] {allow-input: true}
beam_size = 5 #@param {type:"integer"}
initial_prompt = '' #@param {type:"string"}
skip_existing = True #@param {type:"boolean"}
#@markdown skip_existing : docx가 이미 있는 파일은 건너뜀 (연결 끊긴 뒤 이어서 돌릴 때)
download_docx = False #@param {type:"boolean"}
shutdown_after = False #@param {type:"boolean"}
#@markdown shutdown_after : 끝나면 Colab 런타임 종료 (GPU 사용량 절약)

import glob
import os
import shlex
import subprocess
from zipfile import ZipFile

REPO = '/content/whisper-am'

args = [data_dir, '-m', model, '-b', str (beam_size)]
if out_dir        : args += ['-o', out_dir]
if initial_prompt : args += ['-p', initial_prompt]
if skip_existing  : args.append ('--skip-existing')

cmd = ['bash', REPO + '/colab/whisper-am.sh'] + args
print ('터미널 커맨드 :')
print ('  ' + shlex.join (cmd))
print ()

### run the cli and stream its output
proc = subprocess.Popen (cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout :
    print (line, end='')
proc.wait ()
print ('\nexit code :', proc.returncode)

if download_docx and proc.returncode == 0 :
    from google.colab import files
    docx_files = glob.glob (os.path.join (out_dir or data_dir, '*.docx'))
    with ZipFile ('/content/docx.zip', 'w') as zip_obj :
        for doc_file in docx_files :
            zip_obj.write (doc_file, os.path.basename (doc_file))
    files.download ('/content/docx.zip')

if shutdown_after :
    from google.colab import runtime
    print ('Google Colab runtime unassigned.')
    runtime.unassign ()
